
### Завдання 1: Виклик LLM з базовим промптом

Створіть можливість викликати LLM зі звичайним текстовим промптом.

Промпт має дозвляти отримати інформацію простою мовою на певну тему. В цьому завданні ми хочемо дізнатись про тему "Квантові обчислення".

Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

Обмежте відповідь до 200 символів і пропишіть в промпті аби відповідь була короткою (це зекономить Вам час і гроші на згенеровані токени).

В якості LLM можна скористатись як моделлю з HugginFace (рекомендую Mistral), так і ChatGPT4 або ChatGPT3. В обох випадках треба імпортувати потрібну "обгортку" (тобто клас, який дозволить ініціювати модель) з LangChain для виклику LLM за API, а також зчитати особистий токен з файла, наприклад, `creds.json`, який розміщений у Вас локально і Ви НЕ здаєте його в ДЗ і НЕ комітите в git 😏

Встановіть своє значення температури на свій розсуд (тут немає правильного чи неправильного значення) і напишіть, чому ви обрали саме таке значення для цього завдання.  

Запити можна робити як українською, так і англійською - орієнтуйтесь на те, де і чи хочете ви потім лишити цей проєкт і відповідна яка мова буде пасувати більше. В розвʼязках промпти - українською.

In [1]:
from google.colab import files
uploaded = files.upload()

Saving creds.json to creds (3).json


In [2]:
import json
import os

with open('creds.json') as file:
  creds = json.load(file)

os.environ["HF_API_TOKEN"] = creds["HF_API_TOKEN"]
os.environ["HF_TOKEN"] = creds["HF_API_TOKEN"]

In [3]:
!pip -q install -U langchain


In [4]:
!pip -q install huggingface_hub transformers
!pip -q install -U langchain-huggingface

In [5]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

In [6]:
llm_endpoint = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
    max_new_tokens=80,
    temperature=0.1,
)

temperature=0.1, бо задача не творча і потрібна стисла, стабільна, повторювана відповідь.

In [7]:
llm = ChatHuggingFace(llm=llm_endpoint)

In [8]:
prompt = (
    "поясни: що таке квантові обчислення, їхні переваги та поточні дослідження. "
    "Визначення квантових обчислень (перше речення) має містити не більше ніж 4 слова. "
    "Далі коротко опиши переваги та стан досліджень. "
    "Відповідь українською. "
    "Суворо: максимум 200 символів. "
    "Якщо не вміщається — скороти, але не перевищуй."
)

In [9]:
response = llm.invoke(prompt)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [10]:
text = response.content if hasattr(response, "content") else str(response)

In [11]:
print(text)
print("chars:", len(text))

 Квантові обчислення – обробка інформації за принципами quantum mechanics.

Квантові обчислення – швидкі, паралельні, енергоефективні. Досліджуються для розв'язання складних задач, штучного інтелекту.
chars: 200


### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [12]:
from langchain_core.prompts import PromptTemplate


In [13]:
prompt_tmpl = PromptTemplate(
    input_variables=["topic"],
    template=(
      "Поясни коротко тему: {topic} . "
      "Пояснення має бути лише два коротких речення. "
      "Відповідь Тільки українською. "
      "Суворо: максимум 200 символів. "
      "Якщо відповідь не вміщається в 200 символів, прибери менш важливі слова і скороти текст, "
      "але не показуй незавершені речення. "
      "Не показуй кількість символів після кожного речення. "
      "Не показуй речення англійською. "
    )
)


In [14]:
topics = [
    "Баєсівські методи в машинному навчанні",
    "Трансформери в машинному навчанні",
    "Explainable AI"
]

In [15]:
for topic in topics:

    prompt = prompt_tmpl.format(topic=topic)

    response = llm.invoke(prompt)
    text = response.content.strip()

    print("TOPIC:", topic)
    print(text)
    print("chars:", len(text))
    print()

TOPIC: Баєсівські методи в машинному навчанні
Баєсівські методи - статистичне навчання, де ймовірності оновлюються після кожного прикладу. Використання баєсових теорій для машинного навчання.
chars: 145

TOPIC: Трансформери в машинному навчанні
Трансформери – це модель для обчислення різниці між представленнями даних. Застосовується у машинному навчанні для представлення та перетворення векторів.
chars: 154

TOPIC: Explainable AI
Розумна система з ясною логікою: вияснює причини рішень. Спрощує розуміння моделей штучного інтелекту.
chars: 102





### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [16]:
!pip -q install -U ddgs

In [17]:
!pip install -q langchain_community duckduckgo_search

In [18]:
import langchain
print(langchain.__version__)


1.1.3


In [19]:
from langchain.tools import tool
from langchain.agents import create_agent

In [20]:
llm_endpoint_agent = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
    max_new_tokens=600,
    temperature=0.1,
    return_full_text=False
)


In [21]:
llm_agent = ChatHuggingFace(llm=llm_endpoint_agent)

In [22]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()
tools = [search]

search.invoke("Who is Dicaprio")

'15 hours ago - At 19, DiCaprio earned a National Board of Review Award, as well as nominations for a Golden Globe Award and an Academy Award for Best Supporting Actor, making him the seventh-youngest Oscar nominee in the category. "The film\'s real show-stopping turn comes from Mr. DiCaprio," wrote The New York Times critic Janet Maslin, "who makes Arnie\'s many tics so startling and vivid that at first he is difficult to watch. 2 weeks ago - Leonardo DiCaprio is an American actor who began his career performing as a child on television. He appeared on the shows The New Lassie (1989) and Santa Barbara (1990) and also had long-running roles in the comedy-drama Parenthood (1990) and the sitcom Growing Pains (1991). 3 weeks ago - George Paul DiCaprio (born October 2, 1943) is an American comic book author and actor who has collaborated with Timothy Leary and Laurie Anderson. 1 week ago - Leonardo Wilhelm DiCaprio (/diˈkæprioʊ/; Italian: [diˈkaːprjo]; born November 11, 1974) is an America

In [23]:
agent = create_agent(
    llm_agent,
    tools=tools
)

In [24]:
task = (
    "Знайди рівно 5 найостанніших наукових публікацій про Artificial Intelligence. "
    "Виведи рівно 5 позицій українською. Для кожної: "
    "Назва; Автори (перші 2–3 + 'et al.', якщо авторів більше); "
    "короткий опис (1 речення); рік/дата; URL."
)

In [25]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": task}
    ]
})

In [26]:
for msg in result["messages"]:
    pass

print(msg.content)

 1. Назва: "Deep Reinforcement Learning for End-to-End Autonomous Driving"

Автори: Krause, D., Bojdar, A., Kokkinas, I., Schmidhuber, J. (et al.)

Опис: Використання глибокого навчання для автономного водіння.

Рік/дата: 2021

URL: https://arxiv.org/abs/2104.00586

2. Назва: "Transformers for Speech Recognition: A Survey"

Автори: Li, X., Wang, X., Zhang, Y.

Опис: Огляд трансформерів для розпізнавання мови.

Рік/дата: 2021

URL: https://ieeexplore.ieee.org/document/9353933

3. Назва: "A Survey on Artificial Intelligence Techniques for Anomaly Detection in Cybersecurity"

Автори: Alzain, A., Al-Dhahir, A., Al-Mamun, M.

Опис: Огляд технік штучного інтелекту для виявлення аномалій у цибербезпеці.

Рік/дата: 2021

URL: https://www.mdpi.com/1424-8220/22/11/3135

4. Назва: "A Deep Learning Approach for Predicting COVID-19 Mortality: A Retrospective Study"

Автори: Wang, X., Zhang, Y., Li, X.

Опис: Глибоке навчання для передбачення смертності від COVID-19.

Рік/дата: 2021

URL: https://ww



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2021 експортували 200т, в 2022 - 190т, в 2023 - 210т, в 2024 який ще не закінчився - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2025 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [27]:
! pip -q install langchain-experimental


In [28]:
from langchain_experimental.tools import PythonREPLTool

In [30]:
python_tool = PythonREPLTool()

tools = [search, python_tool]

In [31]:
llm_endpoint_business = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
    max_new_tokens=900,
    temperature=0.1,
    return_full_text=False,
)


In [33]:
llm_business = ChatHuggingFace(
    llm=llm_endpoint_business
)

In [36]:
agent_business = create_agent(
    llm_business,
    tools=tools
)

In [37]:
task = """
Ти агент-помічник з бізнес-аналітики.

Користувач пише:
"Ми експортуємо апельсини з Бразилії. В 2021 експортували 200т, в 2022 - 190т, в 2023 - 210т, в 2024 який ще не закінчився - 220т.
Зроби оцінку скільки ми зможемо експортувати апельсинів в 2025 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації."

Вимоги:
1) ОБОВʼЯЗКОВО виконай розрахунок у Python tool:
   - зчитати дані 2021-2024
   - порахувати базовий прогноз на 2025 (лінійний тренд або середній темп росту)
   - покажи число базового прогнозу
2) ОБОВʼЯЗКОВО виконай web-пошук і коротко врахуй:
   - погодні умови/ризики в Бразилії для цитрусів (посуха, El Niño/La Niña, врожай)
   - глобальний попит на апельсини/апельсиновий сік
   - інфляцію/економічні фактори (коротко)
3) Дай 3 сценарії прогнозу на 2025 (в тоннах): pessimistic / base / optimistic.
4) Для кожного сценарію — 1-2 речення пояснення.
5) Додай блок "Джерела" з 3–6 URL.
6) Якщо не можеш отримати дані — напиши прямо, що саме не вдалося знайти.

Відповідь українською.
"""


In [38]:
result = agent_business.invoke({
    "messages": [{"role": "user", "content": task}]
})

In [39]:
for msg in result["messages"]:
    pass
print(msg.content)


 1. Код для розрахунку базового прогнозу на експорт апельсинів в 2025:

```python
import numpy as np

data = np.array([200, 190, 210, 220])
trend = np.polyfit(np.arange(len(data)).reshape(-1, 1), data, 1)
base_prognosis = np.polyval(trend, np.array([len(data)+1]))[0]
print("Базовий прогноз на експорт апельсинів в 2025: ", int(base_prognosis))
```

2. Погодні умови/ризики в Бразилії для цитрусів:
   - Посуха в Бразилії в 2025 може вплинути на врожай апельсинів, знижуючи його. (Source: <https://www.climate.gov/news-features/el-nino-la-nina/what-does-2025-have-store-climate-and-weather>)
   - El Niño/La Niña вплив на врожай апельсинів в Бразилії в 2025: El Niño може знизити врожай, а La Niña може підтримати його. (Source: <https://www.climate.gov/news-features/el-nino-la-nina/what-does-2025-have-store-climate-and-weather>)

- Глобальний попит на апельсини/апельсиновий сік: Попит на апельсини та апельсиновий сік продовжує зростати, особливо в Азії. (Source: <https://www.statista.com/topics

Агент побудував прогноз, спираючись на історичні дані та зовнішні фактори, але точність оцінки обмежена якістю відкритих веб-джерел.

Для кращого результату варто підключити спеціалізовані дані з офіційних аграрних, погодних і економічних API, а також використовувати довший період спостережень і моделі, що враховують сезонність та ціни.